# hist_gradient_boosting — Yaswanth
Read PRE_PROCESSING_DECISIONS.md and docs/TRAINING_GUIDE.md. Run 01 contains measured outputs. Review the results and add your own interpretation before submitting. Complete all five folds before comparing models.

In [1]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Open this notebook from the repository root or notebooks directory"
os.chdir(ROOT)


In [2]:
get_ipython().run_line_magic("run", "notebooks/00_shared_training.ipynb")

In [3]:
MODEL = "hist_gradient_boosting"
RUN_ID = "run_01"


## Optional smoke check
Set RUN_SMOKE=True to check the interface on 200 observations. These are not full experiment results.

In [4]:
RUN_SMOKE = False
if RUN_SMOKE:
    run(MODEL, "processed", f"scratch/{RUN_ID}", folds=[0], final=True, smoke=True)

## Full model comparison
This trains five trip-based CV fits and a final all-training fit, then evaluates student-test data. Preserve warnings and configuration. Use a fresh RUN_ID for another experiment.

In [5]:
run(MODEL, "processed", f"results/{RUN_ID}", folds=[0,1,2,3,4], final=True)

fold_0 {"split": "fold_0", "fit_seconds": 5.176061628997559, "inference_seconds": 0.4430520649984828, "inference_ms_per_observation": 0.015437354181131806, "metrics": {"all_points": {"observations": 28700, "top1_accuracy": 0.7734843205574913, "macro_precision": 0.6442221672431783, "macro_recall": 0.5898166051778505, "macro_f1": 0.6061790279028297, "candidate_miss_rate": 0.00975609756097561, "categorical_log_loss_with_miss_state": 0.9563917531299337}, "reliable_points": {"observations": 20087, "top1_accuracy": 0.8616518146064619, "macro_precision": 0.7726748211400877, "macro_recall": 0.7303527353589847, "macro_f1": 0.745914570640258, "candidate_miss_rate": 0.0010952357245979987, "categorical_log_loss_with_miss_state": 0.37230594503274755}, "reliable_coverage": 0.6998954703832753, "candidate_binary_log_loss": 0.1573843324861123, "route_continuity": null, "route_continuity_reason": "Needs raw road network and an agreed path/connectivity definition; do not invent a value."}, "fit_subset_me

## Inspect measured results

In [6]:
import json
output = Path("results") / RUN_ID / MODEL
print((output / "summary.json").read_text())
records = json.loads((output / "metrics.json").read_text())
for record in records:
    print(record["split"], record["metrics"]["all_points"], record["warnings"])

{
  "model": "hist_gradient_boosting",
  "smoke": false,
  "cv_macro_f1_mean": 0.6470518764970621,
  "cv_macro_f1_std": 0.044133143212714644,
  "completed_folds": 5,
  "student_test_completed": true
}
fold_0 {'observations': 28700, 'top1_accuracy': 0.7734843205574913, 'macro_precision': 0.6442221672431783, 'macro_recall': 0.5898166051778505, 'macro_f1': 0.6061790279028297, 'candidate_miss_rate': 0.00975609756097561, 'categorical_log_loss_with_miss_state': 0.9563917531299337} []
fold_1 {'observations': 40694, 'top1_accuracy': 0.7523959306040202, 'macro_precision': 0.7168024041006016, 'macro_recall': 0.6720993739520688, 'macro_f1': 0.6854207761561497, 'candidate_miss_rate': 0.10217722514375584, 'categorical_log_loss_with_miss_state': 3.09273662593058} []
fold_2 {'observations': 49146, 'top1_accuracy': 0.4649411956212103, 'macro_precision': 0.6310864216947142, 'macro_recall': 0.5692648535993067, 'macro_f1': 0.5887443636271409, 'candidate_miss_rate': 0.28549627640092784, 'categorical_log_l

## Your interpretation
Write your own explanation of the algorithm/objective, hyperparameters, actual results, timing scope, fit and generalisation, convergence where applicable, and proxy-label limitations. Route continuity remains N/A until common graph scoring is implemented. Do not invent loss curves or results.

### Run 01 findings
Histogram Gradient Boosting bins numeric features and adds small decision trees that reduce binary log loss. Tree splits can learn interactions, such as giving heading more weight when the heading flag is available. This model used 150 maximum iterations, 31 maximum leaf nodes, learning rate 0.1 and L2 regularization of 1.

Mean CV arc Macro F1 was 0.6471 (SD 0.0441). On 19,160 student-test points, Top-1 accuracy was 80.78%, arc Macro F1 was 0.6113 and categorical log loss was 0.6710. The five CV fits and final refit took 29.34 seconds of fitting. Student-test classifier inference averaged about 15.0 microseconds per observation in this batch.

### Compare the two baselines
Boosting's mean CV Macro F1 lead was about 0.0042, small relative to fold variability. On student test, boosting made one more correct Top-1 prediction, while logistic had higher arc Macro F1 and boosting had lower categorical log loss. The next three models are needed before choosing a tuning target. Compare meaningful fit and validation subsets before attributing any gap to overfitting. The labels are route-alignment proxies, and inference timing excludes preprocessing and HMM.
